# Publication-style condition-group figure

Rebuild the eight-group figure from the saved 08D results. Run all cells in Antigravity using the project's existing Python environment. This notebook performs **no training, new uncertainty estimation, or manuscript editing**.

The figure uses a compact table–forest layout, embedded serif fonts, exact NIFC category predicates, a common 0–100% scale, and aligned numerical columns. Definitions belong in the supplied LaTeX caption instead of a large title and footer inside the image.

Original group order, all eight observed Held fractions, pointwise intervals, and support counts are preserved. Outputs have a new `_publication` suffix; the earlier figure and `paper.tex` are not overwritten.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "outputs/08_condition_outcomes").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run from the wildfire_fireline_effectiveness project or its notebooks directory.")
PROJECT = ROOT / "overleaf_revision_08_series_20261001"
OUT = PROJECT / "revision_figures_08"
SOURCE = ROOT / "outputs/08_condition_outcomes/08D_20261001T002458_892164Z_f3e697/frozen_rule_weighting_checks.csv"
NAME = "condition_groups_unit_held_rates_publication"
OUT.mkdir(parents=True, exist_ok=True)
assert OUT.resolve().is_relative_to(PROJECT.resolve())

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

protected_paths = [PROJECT / "paper.tex", PROJECT / "supplement.tex",
                   OUT / "condition_groups_unit_held_rates.pdf",
                   OUT / "condition_groups_unit_held_rates.png"]
protected = {p: sha256(p) for p in protected_paths if p.exists()}
source_hash = sha256(SOURCE)
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = "Times New Roman" if "Times New Roman" in available_fonts else "STIXGeneral"
print("Font:", FONT)
print("Saved-result source:", SOURCE)


## Verify the unchanged population and numerical results

Select the combined 2023–2024, equal-fireline-unit summaries, not the equal-fire or source-record-weighted alternatives. The original tree was fitted using 2020–2022 outcomes. Its display groups 1–8 correspond to original leaves 3, 4, 6, 7, 10, 11, 13, and 14.

The type split is exact: groups 1–4 exclude the recorded category **Completed Dozer Line**, while groups 5–8 require that category. The first block must not be renamed “non-dozer,” because the complement of one recorded category is not a verified construction-method classification.


In [ ]:
raw = pd.read_csv(SOURCE)
order = [3, 4, 6, 7, 10, 11, 13, 14]
selected = raw.loc[
    raw["scope"].astype(str).eq("2023_2024")
    & raw["weight"].eq("unit")
    & raw["family"].eq("frozen")
].copy()
assert selected["leaf_id"].is_unique
g = selected.set_index("leaf_id").loc[order].copy()
g.insert(0, "display_group", range(1, 9))
assert len(g) == 8
assert g["n_units"].tolist() == [33339, 12379, 9093, 10382, 1019, 16548, 9454, 30584]
assert g["n_fires"].tolist() == [115, 87, 83, 62, 71, 103, 46, 97]
assert int(g["n_units"].sum()) == 122798
assert np.allclose(g["held_fraction"], g["n_held"] / g["n_units"], atol=1e-14, rtol=0)
assert ((g["ci_lower"] >= 0) & (g["ci_upper"] <= 1)).all()
assert ((g["ci_lower"] <= g["held_fraction"]) & (g["held_fraction"] <= g["ci_upper"])).all()
assert g["conditions"].iloc[:4].str.contains("is not Completed Dozer Line", regex=False).all()
assert g["conditions"].iloc[4:].str.contains("is Completed Dozer Line", regex=False).all()
assert g["n_rule_path_imputed"].eq(0).all()
assert g["n_unseen_type"].eq(0).all()
print("Verified all 8 groups and 122,798 test fireline units.")
display(g[["display_group", "n_units", "n_fires", "held_fraction", "ci_lower", "ci_upper"]])


## Draw the journal-style figure

The design is sized for a two-column paper figure. It intentionally omits an in-image headline and long methodological footnotes: the figure caption supplies the scientific context and definitions. Black markers and interval bars remain legible in grayscale. Counts and estimates occupy aligned columns instead of floating above points. A 50% reference indicates equal recorded Held and Burned Over counts, not chance prediction performance.

Cutoffs below are rounded only for display; no group is reassigned. The label “Recorded type” refers to the exact NIFC attribute category, not a causal treatment.


In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": [FONT],
    "font.size": 10,
    "mathtext.fontset": "stix",
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.unicode_minus": False,
    "savefig.facecolor": "white",
})

conditions = [
    "RH ≤ 35.6%\ngust ≤ 7.81 m/s",
    "RH ≤ 35.6%\ngust > 7.81 m/s",
    "RH > 35.6%\nD ≤ 211 m",
    "RH > 35.6%\nD > 211 m",
    "D ≤ 10.4 m",
    "10.4 m < D ≤ 150 m",
    "D > 150 m\nE ≤ 1,075 m",
    "D > 150 m\nE > 1,075 m",
]
ys = np.array([0.80, 0.72, 0.64, 0.56, 0.40, 0.32, 0.24, 0.16])
fig = plt.figure(figsize=(7.10, 4.65), facecolor="white")
ax = fig.add_axes([0.432, 0, 0.268, 1], facecolor="none")
ax.set_xlim(-2, 102)
ax.set_ylim(0, 1)
ax.set_yticks([])
for side in ["top", "left", "right"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_position(("axes", 0.095))
ax.spines["bottom"].set_bounds(0, 100)
ax.spines["bottom"].set_linewidth(0.65)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(["0", "25", "50", "75", "100"])
ax.tick_params(axis="x", direction="out", length=3, width=0.65, pad=4)

# Draw guides only across data blocks, leaving the type headings uncluttered.
for low, high in [(0.535, 0.835), (0.135, 0.435)]:
    for x in [25, 75]:
        ax.vlines(x, low, high, color="#dddddd", linewidth=0.45, zorder=0)
    ax.vlines(50, low, high, color="#777777", linewidth=0.65,
              linestyles=(0, (2.4, 2.4)), zorder=0)

texts = []
def label(x, y, text, ha="left", bold=False, fontsize=10, **kwargs):
    artist = fig.text(x, y, text, ha=ha, va="center", fontsize=fontsize,
                      fontweight="bold" if bold else "normal",
                      color="#161616", linespacing=1.10, **kwargs)
    texts.append(artist)
    return artist

label(0.022, 0.944, "Group", bold=True)
label(0.125, 0.944, "Conditions", bold=True)
label(0.566, 0.944, "Observed Held\nfraction (%)", ha="center", bold=True)
label(0.766, 0.944, "Held\n(%)", ha="right", bold=True)
label(0.901, 0.944, "Fireline\nunits", ha="right", bold=True)
label(0.985, 0.944, "Fires", ha="right", bold=True)
label(0.022, 0.868, 'Recorded type ≠ “Completed Dozer Line”', bold=True)
label(0.022, 0.468, 'Recorded type = “Completed Dozer Line”', bold=True)

for y, (leaf, row), condition in zip(ys, g.iterrows(), conditions):
    lo, point, hi = 100 * np.array([row.ci_lower, row.held_fraction, row.ci_upper])
    ax.errorbar(point, y, xerr=np.array([[point - lo], [hi - point]]),
                fmt="o", markersize=4.1, markeredgewidth=0.65,
                color="#202020", ecolor="#202020", elinewidth=0.9,
                capsize=2.4, capthick=0.8, zorder=3)
    label(0.061, y, str(int(row.display_group)), ha="right")
    label(0.125, y, condition)
    label(0.766, y, f"{point:.1f}", ha="right")
    label(0.901, y, f"{int(row.n_units):,}", ha="right")
    label(0.985, y, f"{int(row.n_fires):,}", ha="right")

for y, lw, color in [(0.985, 0.8, "#222222"), (0.901, 0.55, "#555555"),
                      (0.510, 0.55, "#888888")]:
    fig.add_artist(Line2D([0.022, 0.985], [y, y], transform=fig.transFigure,
                          linewidth=lw, color=color, clip_on=False))

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
boxes = [(text, text.get_window_extent(renderer)) for text in texts]
canvas = fig.bbox
for text, box in boxes:
    assert box.x0 >= 0 and box.y0 >= 0 and box.x1 <= canvas.x1 and box.y1 <= canvas.y1, text.get_text()
overlaps = []
for j, (left, a) in enumerate(boxes):
    for right, b in boxes[j + 1:]:
        if a.overlaps(b):
            overlaps.append((left.get_text(), right.get_text()))
assert not overlaps, overlaps

pdf_path = OUT / f"{NAME}.pdf"
png_path = OUT / f"{NAME}.png"
fig.savefig(pdf_path, metadata={
    "Title": "Observed Held fractions in fixed condition groups",
    "Subject": "2023–2024 equal-fireline-unit outcomes; 2,000 whole-fire bootstrap intervals",
    "Creator": "08_series_condition_figure_publication.ipynb",
})
fig.savefig(png_path, dpi=400)
plt.close(fig)
display(Image(filename=str(png_path), width=1100))
print(pdf_path)


## Replacement LaTeX figure block

Copy the new PDF into the existing `revision_figures_08` directory in Overleaf. Replace the previous figure block with the following; no manuscript file is changed by this notebook.

```latex
\begin{figure*}[!t]
\centering
\includegraphics[width=\textwidth]{revision_figures_08/condition_groups_unit_held_rates_publication.pdf}
\caption{Observed Held fractions for eight condition groups learned from 2020--2022 and applied unchanged to the combined 2023--2024 test data. Each test fireline unit receives equal weight. Points show observed fractions; bars show pointwise 95\% intervals from 2,000 whole-fire bootstrap resamples. The dashed reference marks equal Held and Burned Over counts. Recorded type is the NIFC category; $\ne$ excludes only the category Completed Dozer Line. RH denotes mean forecast relative humidity over 0--18 hours after prediction time; gust denotes the wind-gust forecast at 18 hours; E denotes mean elevation. D is the closest distance between the mapped fire area and any part of the fireline unit at prediction time. Cutoffs are rounded for display. Counts refer to the combined test data; fires may contribute to several groups.}
\label{fig:conditions}
\end{figure*}
```


In [ ]:
# Save a compact numerical audit and provenance for the regenerated figure.
audit_path = OUT / f"{NAME}_data.csv"
audit = g.reset_index()[[
    "display_group", "leaf_id", "n_units", "n_source_records", "n_fires",
    "n_held", "n_burned_over", "held_fraction", "ci_lower", "ci_upper",
]]
audit.to_csv(audit_path, index=False)
for path, before in protected.items():
    assert sha256(path) == before, f"Protected file changed: {path}"
assert sha256(SOURCE) == source_hash

manifest = {
    "notebook": "notebooks/08_series_condition_figure_publication.ipynb",
    "source": str(SOURCE),
    "source_sha256": source_hash,
    "scope": "2023_2024",
    "evaluation_weight": "unit",
    "training_years": [2020, 2021, 2022],
    "test_years": [2023, 2024],
    "bootstrap_resamples": 2000,
    "group_to_leaf": {str(i + 1): leaf for i, leaf in enumerate(order)},
    "font_family": FONT,
    "figure_size_inches": [7.10, 4.65],
    "body_font_points": 10,
    "embedded_truetype_pdf_fonts": True,
    "all_eight_groups_preserved": True,
    "text_overlap_check_passed": True,
    "no_training_or_statistical_recomputation": True,
    "protected_original_files": {str(p): h for p, h in protected.items()},
    "outputs": {str(p): sha256(p) for p in [pdf_path, png_path, audit_path]},
}
manifest_path = OUT / f"{NAME}_provenance.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Saved vector PDF, 400-dpi PNG, numerical audit, and provenance.")
print("Original figure, source results, and manuscript files remain unchanged.")
